In [1]:
import pandas as pd
fixed_entries = [{
"question": "what is the annual fee",
"answer": "The annual fee is Rs 500.",
"keywords": "fee cost price charge",
"category": "billing"},
    {
"question": "how to reset password",
"answer": "Go to Settings > Reset Password.",
"keywords": "password reset login",
"category": "account"},
{
"question": "what are your working hours",
"answer": "We are open 9 AM to 5 PM.",
"keywords": "hours timing open time",
"category": "general"
    },
    {
"question": "how can i pay the fee",
"answer": "You can pay via UPI, card, or net banking.",
"keywords": "pay payment upi fee",
"category": "billing"
    }
]
roll_number = "1024160057"
categories = ["billing", "account", "general"]
last_two_digits = roll_number[-2:]
personalized_entries = []
d = int(last_two_digits[0])
category = categories[d % 3]
personalized_entries.append({
"question": "how can i contact customer support",
"answer": "You can contact customer support through the help desk or support email.",
"keywords": "support help contact",
"category": category
})
d = int(last_two_digits[1])
category = categories[d % 3]
personalized_entries.append({
"question": "how do i update my registered mobile number",
"answer": "Go to Account Settings > Personal Information and update your mobile number.",
"keywords": "mobile number update",
"category": category
})
all_entries = fixed_entries + personalized_entries
df = pd.DataFrame(all_entries)
print(df)

                                      question  \
0                       what is the annual fee   
1                        how to reset password   
2                  what are your working hours   
3                        how can i pay the fee   
4           how can i contact customer support   
5  how do i update my registered mobile number   

                                              answer                keywords  \
0                          The annual fee is Rs 500.   fee cost price charge   
1                   Go to Settings > Reset Password.    password reset login   
2                          We are open 9 AM to 5 PM.  hours timing open time   
3         You can pay via UPI, card, or net banking.     pay payment upi fee   
4  You can contact customer support through the h...    support help contact   
5  Go to Account Settings > Personal Information ...    mobile number update   

  category  
0  billing  
1  account  
2  general  
3  billing  
4  general  
5  account

In [2]:
def score_query(query, df):
    query_words = set(query.lower().split())
    results = []

    for index, row in df.iterrows():
        keywords = set(row["keywords"].lower().split())

        matched_words = query_words.intersection(keywords)

        score = len(matched_words)

        if score > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "matched_keywords": ", ".join(matched_words),
                "confidence": score
            })


    results = sorted(results, key=lambda x: x["confidence"], reverse=True)

    return pd.DataFrame(results)

query = input("Enter your query: ")

results = score_query(query, df)

print("\nMatching FAQs:")
print(results)

Enter your query: how do i pay my fee

Matching FAQs:
                 question                                      answer  \
0   how can i pay the fee  You can pay via UPI, card, or net banking.   
1  what is the annual fee                   The annual fee is Rs 500.   

  category matched_keywords  confidence  
0  billing         pay, fee           2  
1  billing              fee           1  


In [3]:
def same_category(category_name, df):
    return df[df["category"] == category_name]


category_name = personalized_entries[0]["category"]

result = same_category(category_name, df)

print("Category:", category_name)
print("\nFAQs in this category:")
print(result)

Category: general

FAQs in this category:
                             question  \
2         what are your working hours   
4  how can i contact customer support   

                                              answer                keywords  \
2                          We are open 9 AM to 5 PM.  hours timing open time   
4  You can contact customer support through the h...    support help contact   

  category  
2  general  
4  general  


In [4]:

index = 0

new_keyword = input("Enter a new keyword: ")

df.loc[index, "keywords"] = df.loc[index, "keywords"] + " " + new_keyword

filename = roll_number + "_faq_data.csv"
df.to_csv(filename, index=False)

print("\nUpdated FAQ entry:")
print(df.loc[index])

print("\nDataFrame saved as:", filename)

Enter a new keyword: account

Updated FAQ entry:
question           what is the annual fee
answer          The annual fee is Rs 500.
keywords    fee cost price charge account
category                          billing
Name: 0, dtype: object

DataFrame saved as: 1024160057_faq_data.csv


In [5]:
category_counts = df.groupby("category").size()

print("Number of FAQ entries per category:")
print(category_counts)

Number of FAQ entries per category:
category
account    2
billing    2
general    2
dtype: int64


In [6]:
def score_query_with_ties(query, df):
    query_words = set(query.lower().split())
    results = []

    for index, row in df.iterrows():
        keywords = set(row["keywords"].lower().split())

        matched_words = query_words.intersection(keywords)
        score = len(matched_words)

        if score > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "matched_keywords": ", ".join(matched_words),
                "confidence": score
            })

    if not results:
        print("No matching FAQs found.")
        return

    results = sorted(results, key=lambda x: x["confidence"], reverse=True)

    highest_score = results[0]["confidence"]
    best_matches = [
        result for result in results
        if result["confidence"] == highest_score
    ]

    if len(best_matches) > 1:
        print("Tie found! Multiple FAQs have the highest confidence:")
    else:
        print("Best matching FAQ:")

    print(pd.DataFrame(best_matches))